# Лекција 12 - Смањење историје ћаскања помоћу агентског бележника

Овај нотебоок приказује како управљати контекстом у дугим разговорима користећи Microsoft Agent Framework. Како разговори расту, број токена се повећава — на крају прелази прозор контекста модела. Ово решавамо уз помоћ **шаблона за сумирање контекста** и **агентског бележника** за упорну меморију.

## Шта ћете научити:
1. **Зашто је управљање контекстом важно**: Разумевање лимита токена и прозора контекста  
2. **Агенти свесни контекста**: Израда агената који управљају својим сопственим контекстом разговора  
3. **Шаблон за сумирање контекста**: Коришћење алата за кондензацију историје разговора  
4. **Агентски бележник**: Упорна меморија која опстаје кроз смањење контекста  

## Предуслови:
- Подешен Azure OpenAI са конфигурисаним променљивима окружења  
- Разумевање основних концепата агената из претходних лекција


## Подешавање


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Зашто је управљање контекстом важно

Свако LLM има ограничени **прозор контекста** — максимални број токена које може обрадити у једном захтеву. Како се вишекратна конверзација одвија:

- **Број токена расте линеарно** са сваким корисничким поруком и одговором асистента.
- **Токени у упиту доминирају трошком** јер се цео историјат шаље поново у сваком кругу.
- На крају конверзација **прелази прозор контекста** и модел или скраћује, или даје грешку.

### Стратегије за управљање контекстом

| Стратегија | Како функционише | Компромис |
|---|---|---|
| **Скраћивање** | Одбаци најстарије поруке | Губи се рани контекст |
| **Сажимање** | Сажима старије поруке у резиме | Нешто детаља се губи, али се кључне тачке задржавају |
| **Бележница / Спољна меморија** | Чува кључне чињенице ван конверзације | Захтева позиве алата, али опстаје при сваком смањењу |

У овом бележнику комбинујемо **сажимање** са алатом **бележнице** како би агент могао да одржи континуитет чак и када се историја конверзације сведе.


## Креирање агента свесног контекста


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Симулирање дугог разговора

Хајде да прођемо кроз разговор са више корака како бисмо видели како се контекст акумулира. Агенат треба да задржи кључне детаље (преференције, буџет, датуме путовања) кроз све кораке и покаже континуитет.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Обратите пажњу како агент задржава контекст из претходних интеракција — он зна о Јапану, сушију, храмовима, фотографији, буџету од 3000 долара, соло путовању и априлском временском периоду. У кратком разговору ово добро функционише, али како разговор расте, поновно слање целе историје постаје скупо.

Наставимо разговор са више интеракција да видимо акумулацију контекста:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Образац за сумирање контекста

Како разговор расте, можемо користити **алат за сумирање** да саберемо акумулирани контекст у компактни формат. Аутомат позива овај алат да забележи кључне преференције тако да, чак и ако старије поруке буду уклоњене, суштинске информације буду сачуване.

Овај образац представља градивни блок за софистицираније смањење историје:
1. Аутомат идентификује кључне чињенице из разговора
2. Позива алат за сумирање да их сачува
3. Старије поруке се могу безбедно уклонити јер резиме бележи шта је важно

Испод дефинишемо алат `summarize_preferences` који аутомат може позвати да забележи компактни резиме онога што је научио.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Резиме

У овом лекцији сте научили како да управљате контекстом у дуготрајним разговорима агената користећи Microsoft Agent Framework:

### Кључни појмови
- **Прозори контекста су ограничени** — сваки токен у историји разговора кошта и урачунава се у лимит.
- **Алати за сумирање** омогућавају агенту да кондензује акумулирани контекст у компактне резимеје, смањујући коришћење токена уз очување суштинских информација.
- **Агентски бележници (scratchpads)** пружају трајну спољашњу меморију која опстаје уз било какво смањење разговора.

### Шта сте изградили
- **Агента свестан контекста** који одржава континуитет у разговорима са више рунди
- **Алат за сумирање** (`summarize_preferences`) који бележи кључне детаље корисника у компактном формату
- **Разговор са више рунди** који демонстрира задржавање контекста и руковање променама

### Примена у стварном свету
- **Ботови за корисничку подршку**: памте преференције током дугих сесија подршке
- **Лични асистенти**: прате текуће пројекте без поновног објашњавања контекста
- **Образовни тутори**: одржавају напредак студената кроз бројне интеракције

### Следећи кораци
- Имплементирати потпуни алат за бележнице са перзистенцијом базираном на фајловима
- Додати аутоматско скраћивање историје након сумирања
- Комбиновати са векторским базама података за семантичко претраживање меморије
- Изградити агенте који могу наставити разговоре данима касније са пуним контекстом


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Пажња**:  
Овај документ је преведен коришћењем AI преводилачке услуге [Co-op Translator](https://github.com/Azure/co-op-translator). Иако настојимо да превод буде тачан, имајте на уму да аутоматизовани преводи могу садржати грешке или нетачности. Оригинални документ на његовном изворном језику треба сматрати ауторитетом. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која могу настати коришћењем овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
